# Multi-Regional AeroMAPS - All Regions (Simple)

This notebook demonstrates running multiple regions sequentially using `MultiRegionalProcess`.

## Configuration

- **Mode**: `separate_processes` (sequential execution)
- **Regions**: 20 world regions (excluding France and Germany detailed configs)
- **Settings**: Default AeroMAPS parameters for all regions

## 0. Prepare Region Data

Generate partitioning files and config.yaml for each region from their CSV data.

In [ ]:
from pathlib import Path
from aeromaps.utils.functions import create_partitioning
import matplotlib.pyplot as plt
%matplotlib widget
import shutil

# Reference files from region_default
default_folder = Path("data/region_default")
energy_carriers_file = default_folder / "energy_carriers_data.yaml"

# All region folders (with actual folder names including typos and asterisks)
region_folders = [
    "region_af_dom", "region_af_int", "region_as_dom", "region_bras_dom",
    "region_eu_dom*", "region_eu_int*", "region_oc_dom", "region_oc_int",
    "region_other_as_int", "region_other_eur_dom", "region_other_eur_int",
    "region_other_nam_dom", "region_other_nam_int", "region_other_sam_dom",
    "region_sin_int", "region_uk_dom*", "region_uk_int*",
    "region_usa_dom", "region_usa_int", "region_sam_int"
]

data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name
    
    if not folder_path.exists():
        print(f"⚠ Folder not found: {folder_name}")
        continue
    
    csv_file = folder_path / "dataframe_aeromaps.csv"
    if not csv_file.exists():
        print(f"⚠ CSV not found: {csv_file}")
        continue
    
    # 1. Generate partitioning from CSV (creates partitioning_updated_inputs.json)
    create_partitioning(file=str(csv_file), path=str(folder_path))
        
    print(f"✓ Prepared: {folder_name}")

print(f"\n✓ All {len(region_folders)} regions prepared")

## 1. Create Multi-Regional Process

In [ ]:
from aeromaps import create_multi_regional_process

# Create the multi-regional process
process = create_multi_regional_process(
    configuration_file="regionalisation_all_regions.yaml", disable_execution_statistics=True
)

print(f"✓ Process created")
print(f"  Mode: {process.execution_mode}")
print(f"  Regions ({len(process.list_regions())}): {process.list_regions()}")

## 2. Execute Sequential Computation

Run all regions sequentially with progress bar.

In [ ]:
import time

# Sequential execution (default)
start_time = time.time()
process.compute(parallel=False)  # Sequential mode
elapsed = time.time() - start_time

print(f"\n✓ Computation complete in {elapsed:.2f} seconds")

## 2.5. LTAG Reference (scaled on 2019)

Construction d'une série LTAG annuelle régulière, puis calibration multiplicative pour faire correspondre la valeur 2019 à `overall:co2_emissions_including_energy`.
Le facteur multiplicatif est imprimé dans la cellule suivante.

In [ ]:
import pandas as pd
import numpy as np

# LTAG points (year_float, value)
years = [
    2014.9164012738854, 2015.9195859872611, 2016.8949044585988, 2017.3964968152868,
    2017.8980891719746, 2018.316082802548, 2018.6226114649683, 2018.845541401274,
    2018.9291401273886, 2019.0127388535034, 2019.1520700636943, 2019.2914012738854,
    2019.4585987261148, 2019.7093949044588, 2019.9880573248408, 2020.4339171974523,
    2020.9355095541403, 2021.4928343949045, 2022.0501592356688, 2022.6074840764331,
    2022.9140127388537, 2023.4992038216562, 2024.9482484076434, 2027.6234076433122,
    2029.3232484076434, 2031.3017515923568, 2033.0573248407645, 2035.2866242038217,
    2037.125796178344, 2038.90923566879, 2040.4140127388537, 2041.361464968153,
    2042.6154458598728, 2043.9251592356688, 2045.1234076433122, 2046.0151273885351,
    2047.0740445859874, 2048.2444267515925, 2049.2476114649685, 2050.0
]
values = [
    780.0387596899222, 813.9534883720928, 862.4031007751937, 886.627906976744,
    910.8527131782944, 915.6976744186045, 915.6976744186045, 915.6976744186045,
    915.6976744186045, 915.6976744186045, 862.4031007751937, 794.5736434108528,
    726.7441860465115, 620.1550387596899, 508.7209302325582, 547.4806201550387,
    586.2403100775196, 663.7596899224804, 746.1240310077517, 838.1782945736434,
    891.4728682170542, 920.5426356589146, 1017.4418604651162, 1172.4806201550384,
    1259.68992248062, 1356.5891472868216, 1438.9534883720928, 1545.5426356589146,
    1623.0620155038757, 1715.1162790697672, 1792.6356589147285, 1845.9302325581393,
    1918.6046511627906, 1986.4341085271317, 2059.108527131783, 2126.937984496124,
    2189.9224806201546, 2272.286821705426, 2330.426356589147, 2383.7209302325577
]

LTAG_series = pd.Series(values, index=years, name="LTAG_series")

# Regularize on integer years with linear interpolation
years_int = np.arange(int(np.ceil(LTAG_series.index.min())), int(np.floor(LTAG_series.index.max())) + 1)
LTAG_series_yearly = (
    LTAG_series
    .sort_index()
    .reindex(LTAG_series.index.union(years_int))
    .interpolate(method="index")
    .reindex(years_int)
)
LTAG_series_yearly.index.name = "year"
LTAG_series_yearly.name = "LTAG_series_yearly"


# Using CORSIA Methodology to include LCA reduction in combustion-only emissions
# All AeroMAPS emissions are LCA, we have to convert using CORSIA factor. 
corsia_ratio = 3.83 / 3.16 # typical EF for fossil kerosene (LCA/Combustion)

# Next, we want to have both secnarios start from the same basis, so we compute an correction factor for 2019. 
# It should be close to one in theory. 
# Multiplicative calibration so that LTAG(2019) = process overall CO2 incl. energy (2019)
target_2019 = process.data["vector_outputs"]["overall:co2_emissions_including_energy"].loc[2019] / corsia_ratio
ltag_2019 = LTAG_series_yearly.loc[2019]
LTAG_multiplicative_factor = 1.0 #target_2019 / ltag_2019
LTAG_series_yearly_scaled = LTAG_series_yearly * LTAG_multiplicative_factor
LTAG_series_yearly_scaled.name = "LTAG_ref_scaled"

print(f"AeroMAPS vs ATAG 2019: {85/(target_2019 / ltag_2019):.6f}")
print(f"LTAG scaled 2019: {LTAG_series_yearly_scaled.loc[2019]:.6f} | Target 2019: {target_2019:.6f}")

LTAG_series_yearly

In [ ]:
import pandas as pd
# LTAG reference 2: regularize to integer years + calibrate on 2019
ltag2_years = [
    2022.96974522293, 2024.8367834394905, 2026.4808917197454, 2028.125,
    2029.7133757961785, 2031.5246815286625, 2033.0294585987263, 2034.673566878981,
    2035.704617834395, 2038.3240445859874, 2039.9960191082803, 2042.50398089172,
    2043.9808917197454, 2045.6528662420383, 2047.408439490446, 2048.8017515923566,
    2049.9721337579617, 2050.0
]
ltag2_values = [
    896.3178294573643, 983.5271317829456, 1056.2015503875966, 1124.0310077519378,
    1187.0155038759688, 1245.15503875969, 1293.6046511627906, 1346.8992248062013,
    1366.2790697674418, 1443.798449612403, 1497.0930232558137, 1584.302325581395,
    1647.286821705426, 1715.1162790697672, 1787.7906976744184, 1860.4651162790697,
    1913.7596899224804, 1918.6
]

LTAG_series_2 = pd.Series(ltag2_values, index=ltag2_years, name="LTAG_series_2")

years_int_2 = np.arange(int(np.ceil(LTAG_series_2.index.min())), int(np.floor(LTAG_series_2.index.max())) + 1)
LTAG_series_2_yearly = (
    LTAG_series_2
    .sort_index()
    .reindex(LTAG_series_2.index.union(years_int_2))
    .interpolate(method="index")
    .reindex(years_int_2)
)
LTAG_series_2_yearly.index.name = "year"
LTAG_series_2_yearly.name = "LTAG_series_2_yearly"

LTAG_series_2_yearly_scaled = LTAG_series_2_yearly * LTAG_multiplicative_factor
LTAG_series_2_yearly_scaled.name = "LTAG_ref_2_scaled"

LTAG_series_2_yearly_scaled.head()


# LTAG reference 3: regularize to integer years + calibrate on 2019
ltag3_years = [
    2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046,
    2047, 2048, 2049, 2050
]
ltag3_values = [
    895.8689352849383, 879.8734112599909, 909.3399251293149, 926.4981016031657,
    926.5012098379698, 936.2148417933402, 949.3633580267451, 957.1249186598097,
    957.1249308599911, 957.126052337891, 945.4945873653428, 934.203646469638,
    926.0936045559185, 941.386012517454, 881.0343944047679, 812.1982672463873,
    734.5909004402608, 646.9878765583012, 583.3624479511473, 487.22211344836114,
    403.54393795122496, 325.1035203928527, 256.017664016455, 188.46572371929733,
    133.87174643875778, 90.251984364907, 54.74030128095865, 9.689922480620226
]

LTAG_series_3 = pd.Series(ltag3_values, index=ltag3_years, name="LTAG_series_3")

years_int_3 = np.arange(int(np.ceil(LTAG_series_3.index.min())), int(np.floor(LTAG_series_3.index.max())) + 1)
LTAG_series_3_yearly = (
    LTAG_series_3
    .sort_index()
    .reindex(LTAG_series_3.index.union(years_int_3))
    .interpolate(method="index")
    .reindex(years_int_3)
)
LTAG_series_3_yearly.index.name = "year"
LTAG_series_3_yearly.name = "LTAG_series_3_yearly"

LTAG_series_3_yearly_scaled = LTAG_series_3_yearly * LTAG_multiplicative_factor
LTAG_series_3_yearly_scaled.name = "LTAG_ref_3_scaled"

LTAG_series_3_yearly_scaled = LTAG_series_3_yearly_scaled.reindex(
    range(2019, 2051)
)

LTAG_series_3_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 4 (Future Aircraft): regularize to integer years + calibrate on 2019
ltag4_years = [
    2022.9418789808917, 2023.7778662420383, 2025.4498407643314, 2026.954617834395,
    2028.5987261146497, 2030.9673566878982, 2033.0573248407645, 2034.8686305732485,
    2037.2372611464968, 2039.187898089172, 2040.8877388535034, 2042.531847133758,
    2044.2316878980894, 2045.875796178344, 2047.8264331210191, 2050.0
]
ltag4_values = [
    896.3178294573643, 925.3875968992247, 1007.751937984496, 1080.426356589147,
    1143.410852713178, 1230.6201550387595, 1293.6046511627906, 1342.0542635658915,
    1400.1937984496124, 1443.798449612403, 1472.8682170542634, 1506.782945736434,
    1540.6976744186045, 1574.612403100775, 1623.0620155038757, 1686.0465116279067
]

LTAG_series_4 = pd.Series(ltag4_values, index=ltag4_years, name="LTAG_series_4")

years_int_4 = np.arange(int(np.ceil(LTAG_series_4.index.min())), int(np.floor(LTAG_series_4.index.max())) + 1)
LTAG_series_4_yearly = (
    LTAG_series_4
    .sort_index()
    .reindex(LTAG_series_4.index.union(years_int_4))
    .interpolate(method="index")
    .reindex(years_int_4)
)
LTAG_series_4_yearly.index.name = "year"
LTAG_series_4_yearly.name = "LTAG_series_4_yearly"

LTAG_series_4_yearly_scaled = LTAG_series_4_yearly * LTAG_multiplicative_factor
LTAG_series_4_yearly_scaled.name = "LTAG_ref_4_scaled"

LTAG_series_4_yearly_scaled = LTAG_series_4_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_4_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 5 (Ops): regularize to integer years + calibrate on 2019
ltag5_years = [
    2022.8861464968154, 2023.6664012738854, 2025.3383757961785, 2027.7627388535034,
    2029.3232484076434, 2030.9952229299365, 2032.5278662420383, 2033.9769108280257,
    2035.9832802547771, 2037.5437898089174, 2039.8566878980894, 2041.5286624203823,
    2043.7579617834397, 2046.2937898089174, 2048.272292993631, 2050.0
]
ltag5_values = [
    896.3178294573643, 920.5426356589146, 993.2170542635656, 1080.426356589147,
    1133.720930232558, 1182.1705426356586, 1220.9302325581396, 1254.84496124031,
    1293.6046511627906, 1312.984496124031, 1342.0542635658915, 1371.124031007752,
    1395.3488372093022, 1429.2635658914728, 1472.8682170542634, 1506.782945736434
]

LTAG_series_5 = pd.Series(ltag5_values, index=ltag5_years, name="LTAG_series_5")

years_int_5 = np.arange(int(np.ceil(LTAG_series_5.index.min())), int(np.floor(LTAG_series_5.index.max())) + 1)
LTAG_series_5_yearly = (
    LTAG_series_5
    .sort_index()
    .reindex(LTAG_series_5.index.union(years_int_5))
    .interpolate(method="index")
    .reindex(years_int_5)
)
LTAG_series_5_yearly.index.name = "year"
LTAG_series_5_yearly.name = "LTAG_series_5_yearly"

LTAG_series_5_yearly_scaled = LTAG_series_5_yearly * LTAG_multiplicative_factor
LTAG_series_5_yearly_scaled.name = "LTAG_ref_5_scaled"

LTAG_series_5_yearly_scaled = LTAG_series_5_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_5_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]


# LTAG reference 6 (Energy): regularize to integer years + calibrate on 2019
ltag6_years = [
    2023.0, 2024.1679936305734, 2025.5891719745223, 2027.2054140127389,
    2028.9331210191083, 2030.7165605095543, 2032.4164012738854, 2033.7261146496817,
    2034.9243630573249, 2035.8996815286625, 2037.125796178344, 2038.546974522293,
    2039.940286624204, 2040.859872611465, 2041.8351910828026, 2043.0891719745223,
    2044.454617834395, 2045.597133757962, 2046.8232484076434, 2048.4116242038217,
    2050.0
]
ltag6_values = [
    901.1627906976744, 939.9224806201548, 993.2170542635656, 1041.6666666666665,
    1080.426356589147, 1099.8062015503874, 1094.9612403100773, 1075.581395348837,
    1041.6666666666665, 1007.751937984496, 949.612403100775, 872.0930232558139,
    794.5736434108528, 741.2790697674418, 683.1395348837209, 605.6201550387595,
    537.7906976744184, 494.18604651162786, 455.42635658914696, 421.5116279069766,
    402.13178294573663
]

LTAG_series_6 = pd.Series(ltag6_values, index=ltag6_years, name="LTAG_series_6")

years_int_6 = np.arange(int(np.ceil(LTAG_series_6.index.min())), int(np.floor(LTAG_series_6.index.max())) + 1)
LTAG_series_6_yearly = (
    LTAG_series_6
    .sort_index()
    .reindex(LTAG_series_6.index.union(years_int_6))
    .interpolate(method="index")
    .reindex(years_int_6)
)
LTAG_series_6_yearly.index.name = "year"
LTAG_series_6_yearly.name = "LTAG_series_6_yearly"

LTAG_series_6_yearly_scaled = LTAG_series_6_yearly * LTAG_multiplicative_factor
LTAG_series_6_yearly_scaled.name = "LTAG_ref_6_scaled"

LTAG_series_6_yearly_scaled = LTAG_series_6_yearly_scaled.reindex(range(2019, 2051))
LTAG_series_6_yearly_scaled.loc[2019:2022] = LTAG_series_yearly_scaled.loc[2019:2022]

LTAG_series_6_yearly_scaled.head()

## 3. View Outputs

All outputs are stored with namespace prefixes in `data["vector_outputs"]`.

In [ ]:
# Global aggregated outputs
global_outputs = process.get_global_outputs()
print(f"Global outputs shape: {global_outputs.shape}")
global_outputs.head()

In [ ]:
# All outputs with namespaces
print(f"Total columns in vector_outputs: {len(process.data['vector_outputs'].columns)}")
process.data['vector_outputs'].columns.tolist()[:20]  # First 20

In [ ]:
# Regional outputs for a specific region
eu_dom = process.get_regional_outputs("EU_DOM")
print(f"EU_DOM outputs shape: {eu_dom.shape}")
eu_dom.head()

## 3.5. Alternative Scenario: Without New Generation Aircraft

Run a scenario where only fleet renewal with recent aircraft exists (no new generation aircraft like 2035 models).

> **⚠️ Provisional workaround**: This approach uses a temporary swap of the `fleet.yaml` file. 
> An issue is open to enable a cleaner override at the multi-regional configuration level.
> See: https://github.com/AeroMAPS/AeroMAPS/issues/XXX

In [ ]:
import shutil
from pathlib import Path
from aeromaps import create_multi_regional_process

# Paths
fleet_dir = Path("data/fleet")
fleet_original = fleet_dir / "fleet.yaml"
fleet_no_ng = fleet_dir / "fleet_no_ng.yaml"
fleet_backup = fleet_dir / "fleet_backup.yaml"

# 1. Backup the original fleet.yaml
shutil.copy(fleet_original, fleet_backup)
print(f"✓ Backed up fleet.yaml → fleet_backup.yaml")

# 2. Replace with fleet_no_ng.yaml (no new generation aircraft)
shutil.copy(fleet_no_ng, fleet_original)
print(f"✓ Replaced fleet.yaml with fleet_no_ng.yaml")

try:
    # 3. Create and run the process WITHOUT new generation aircraft
    process_no_ng = create_multi_regional_process(
        configuration_file="regionalisation_all_regions.yaml", 
        disable_execution_statistics=True
    )
    
    start_time = time.time()
    process_no_ng.compute(parallel=False)
    elapsed = time.time() - start_time
    print(f"\n✓ Scénario 'No NG' terminé en {elapsed:.2f} secondes")
    
finally:
    # 4. Restore original fleet.yaml
    shutil.copy(fleet_backup, fleet_original)
    print(f"✓ Restored original fleet.yaml")

## 4. Emissions Analysis

Compare CO2 emissions metrics across all regions.

In [ ]:
process.data["vector_outputs"]["overall:co2_emissions_including_aircraft_efficiency"]

In [ ]:
import numpy as np


region_to_continent = {
    "AF_DOM": "Africa",
    "AF_INT": "Africa",
    "AS_DOM": "Asia",
    "OTHER_AS_INT": "Asia",
    "SIN_INT": "Asia",
    "EU_DOM": "Europe",
    "EU_INT": "Europe",
    "UK_DOM": "Europe",
    "UK_INT": "Europe",
    "OTHER_EUR_DOM": "Europe",
    "OTHER_EUR_INT": "Europe",
    "USA_DOM": "North America",
    "USA_INT": "North America",
    "OTHER_NAM_DOM": "North America",
    "OTHER_NAM_INT": "North America",
    "BRAS_DOM": "South America",
    "OTHER_SAM_DOM": "South America",
    "SAM_INT": "South America",
    "OC_DOM": "Oceania",
    "OC_INT": "Oceania",
}


# ── Figure 1: Global CO₂ Emissions – Fill-between Waterfall ──
fig1, ax1 = plt.subplots(figsize=(12, 6))

vo = process.data["vector_outputs"]
vo_no_ng = process_no_ng.data["vector_outputs"]
years = vo.index

# --- Retrieve each successive emission stage (Mt) ---
s_bau = vo["overall:co2_emissions_2019technology"] / corsia_ratio \
    if "overall:co2_emissions_2019technology" in vo.columns else None

s_fleet = vo_no_ng["overall:co2_emissions_including_aircraft_efficiency"] / corsia_ratio \
    if "overall:co2_emissions_including_aircraft_efficiency" in vo_no_ng.columns else None

s_future = vo["overall:co2_emissions_including_aircraft_efficiency"] / corsia_ratio \
    if "overall:co2_emissions_including_aircraft_efficiency" in vo.columns else None

s_ops = vo["overall:co2_emissions_including_load_factor"] / corsia_ratio \
    if "overall:co2_emissions_including_load_factor" in vo.columns else None

s_energy = vo["overall:co2_emissions_including_energy"] / corsia_ratio \
    if "overall:co2_emissions_including_energy" in vo.columns else None

s_net = None
if "overall:co2_emissions_including_energy" in vo.columns and "overall:carbon_offset" in vo.columns:
    s_net = (vo["overall:co2_emissions_including_energy"] - vo["overall:carbon_offset"]) / corsia_ratio

# --- Fill-between wedges ---
fills = [
    (s_bau,    s_fleet,  "#FFE082", 0.55, "Fleet Renewal"),         # yellow light
    (s_fleet,  s_future, "#FFC107", 0.55, "Future Aircraft"),       # yellow deeper
    (s_future, s_ops,    "#FFB74D", 0.55, "Operations"),            # orange
    (s_ops,    s_energy, "#66BB6A", 0.50, "Low Carbon Energy"),     # green
    (s_energy, s_net,    "#BDBDBD", 0.45, " CORSIA Offsets"),               # grey
]

for top, bottom, color, alpha, label in fills:
    if top is not None and bottom is not None:
        ax1.fill_between(years, top, bottom, alpha=alpha, color=color,
                         label=label, edgecolor=color, linewidth=0.5)

# Thin boundary lines at top (BAU) and bottom (net)
if s_bau is not None:
    ax1.plot(years, s_bau, color="#E0A800", linewidth=1, alpha=0.6)
if s_net is not None:
    ax1.plot(years, s_net, color="#757575", linewidth=1.5, alpha=0.8)

# --- ATAG reference lines (black, thin, different linestyles) ---
if "LTAG_series_yearly_scaled" in globals():
    LTAG_series_yearly_scaled.plot(
        ax=ax1, label="ATAG - Frozen tech.", linewidth=1.2, color="black", linestyle=":"
    )
if "LTAG_series_2_yearly_scaled" in globals():
    LTAG_series_2_yearly_scaled.plot(
        ax=ax1, label="ATAG - Renewal", linewidth=1.2, color="black", linestyle="-."
    )
if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(
        ax=ax1, label="ATAG - All measures", linewidth=1.2, color="black", linestyle="--"
    )
if "LTAG_series_4_yearly_scaled" in globals():
    LTAG_series_4_yearly_scaled.plot(
        ax=ax1, label="ATAG - Future Aircraft", linewidth=1.2, color="black", linestyle=(0, (3, 1, 1, 1))
    )
if "LTAG_series_5_yearly_scaled" in globals():
    LTAG_series_5_yearly_scaled.plot(
        ax=ax1, label="ATAG - Ops", linewidth=1.2, color="black", linestyle=(0, (5, 2, 1, 2))
    )
if "LTAG_series_6_yearly_scaled" in globals():
    LTAG_series_6_yearly_scaled.plot(
        ax=ax1, label="ATAG - Energy", linewidth=1.2, color="black", linestyle=(0, (1, 1))
    )

ax1.set_xlabel("Year")
ax1.set_ylabel("CO₂ Emissions (Mt)")
ax1.set_title("Global CO₂ Emissions — Scenario Decomposition")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: CO₂ Evolution — Overview + Continent subplots (normalized, base 1 = 2019) ──
import matplotlib.gridspec as gridspec

vo = process.data["vector_outputs"]
regions = process.list_regions()

continent_palette = {
    "Africa":        "#e41a1c",
    "Asia":          "#377eb8",
    "Europe":        "#4daf4a",
    "North America": "#984ea3",
    "South America": "#ff7f00",
    "Oceania":       "#a65628",
}

region_ls_cycle = ["--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 2, 1, 2))]
region_markers_cycle = ["o", "s", "^", "D", "v", "P", "X", "*"]
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]

# --- Helper: compute net CO₂ for a list of region ids, normalized to base 1 in 2019 ---
def _net_series(region_ids, normalize=True):
    total = None
    for rid in region_ids:
        c1 = f"{rid}:co2_emissions_including_energy"
        c2 = f"{rid}:carbon_offset"
        if c1 in vo.columns and c2 in vo.columns:
            s = vo[c1] - vo[c2]
            total = s if total is None else total + s
    if total is None:
        return None
    if normalize:
        b = total.loc[2019]
        return total / b if b != 0 else None
    return total

# --- Layout: 1 big overview on top, 6 continent subplots below (2 rows × 3 cols) ---
fig2 = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(3, 3, figure=fig2, height_ratios=[2.3, 1, 1],
                       hspace=0.2, wspace=0.20)
ax_main = fig2.add_subplot(gs[0, :])

# ── Main panel: GLOBAL, DOM, INT + all continents ──
# Global
norm_global = _net_series(regions)
if norm_global is not None:
    ax_main.plot(norm_global.index, norm_global, color="black",
                 linewidth=3, label="GLOBAL")

# Aggregated DOM / INT
dom_regions = [r for r in regions if "DOM" in r]
int_regions = [r for r in regions if "INT" in r]
norm_dom = _net_series(dom_regions)
norm_int = _net_series(int_regions)
if norm_dom is not None:
    ax_main.plot(norm_dom.index, norm_dom, color="black",
                 linewidth=2, linestyle="--", label="All DOM")
if norm_int is not None:
    ax_main.plot(norm_int.index, norm_int, color="black",
                 linewidth=2, linestyle=":", label="All INT")

# Each continent
for continent in continents_order:
    c_color = continent_palette[continent]
    c_regs = [r for r in regions if region_to_continent.get(r) == continent]
    norm_c = _net_series(c_regs)
    if norm_c is not None:
        ax_main.plot(norm_c.index, norm_c, color=c_color,
                     linewidth=1.5, label=continent)

# ATAG ref 3
if "LTAG_series_3_yearly_scaled" in globals():
    ltag3_b = LTAG_series_3_yearly_scaled.loc[2019]
    if ltag3_b != 0:
        ax_main.plot(LTAG_series_3_yearly_scaled.index,
                     LTAG_series_3_yearly_scaled / ltag3_b,
                     color="black", linewidth=1.2, linestyle="-.",
                     label="ATAG ref 3")

ax_main.set_ylabel("Relative CO₂ (base 1 = 2019)")
ax_main.set_title("CO₂ Emissions Evolution — Overview", fontsize=12)
ax_main.legend(loc="upper left", fontsize=8, ncol=2)
ax_main.grid(True, alpha=0.3)
ax_main.axhline(y=1, color="gray", linestyle="--", alpha=0.5)

# ── Continent sub-panels ──
for ci, continent in enumerate(continents_order):
    row = 1 + ci // 3
    col = ci % 3
    ax_sub = fig2.add_subplot(gs[row, col])

    c_color = continent_palette[continent]
    c_regs = [r for r in regions if region_to_continent.get(r) == continent]

    # Continent total (thick)
    norm_c = _net_series(c_regs)
    if norm_c is not None:
        ax_sub.plot(norm_c.index, norm_c, color=c_color,
                    linewidth=2.5, label=continent)

    # Individual regions (thin + markers)
    for i, rid in enumerate(c_regs):
        norm_r = _net_series([rid])
        if norm_r is not None:
            ls = region_ls_cycle[i % len(region_ls_cycle)]
            mk = region_markers_cycle[i % len(region_markers_cycle)]
            ax_sub.plot(norm_r.index, norm_r, color=c_color,
                        linewidth=0.9, linestyle=ls, alpha=0.65,
                        marker=mk, markersize=3, markevery=5,
                        label=rid)
            
    if "LTAG_series_3_yearly_scaled" in globals():
        ltag3_b = LTAG_series_3_yearly_scaled.loc[2019]
        if ltag3_b != 0:
            ax_sub.plot(LTAG_series_3_yearly_scaled.index,
                        LTAG_series_3_yearly_scaled / ltag3_b,
                        color="black", linewidth=1.2, linestyle="-.",
                        label="ATAG ref 3")

    ax_sub.set_title(continent, fontsize=10, color=c_color, fontweight="bold")
    ax_sub.legend(fontsize=6, loc="best")
    ax_sub.grid(True, alpha=0.25)
    ax_sub.axhline(y=1, color="gray", linestyle="--", alpha=0.4)
    ax_sub.set_ylabel("Relative CO₂")
    if row == 2:
        ax_sub.set_xlabel("Year")

plt.show()

In [ ]:
# ── Figure 3: ETS Allowances & CORSIA Credits (by continent) ──
fig3, ax3 = plt.subplots(figsize=(10, 6))

vo = process.data["vector_outputs"]
regions = process.list_regions()

# Palette: color = continent
continent_palette = {
    "Africa":        "#e41a1c",
    "Asia":          "#377eb8",
    "Europe":        "#4daf4a",
    "North America": "#984ea3",
    "South America": "#ff7f00",
    "Oceania":       "#a65628",
}
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]

# Linestyle = type of measure
ls_corsia = "-"
ls_ets_eu = ":"
ls_ets_uk = "-."
ls_sum = "--"

# Collect lines grouped by continent: { continent: [(handle, label), ...] }
legend_groups = {c: [] for c in continents_order}
legend_totals = []  # for global totals at the end

# --- ETS per region (EU_DOM and UK_DOM separately) ---
ets_ls = {"EU_DOM": ls_ets_eu, "UK_DOM": ls_ets_uk}
ets_labels = {"EU_DOM": "EU ETS", "UK_DOM": "UK ETS"}
ets_total = None
for region_id in ["EU_DOM", "UK_DOM"]:
    col_fossil = f"{region_id}:energy_consumption_kerosene"
    if col_fossil not in vo.columns:
        continue
    fossil_use = vo[col_fossil].loc[2019:]
    net_co2 = (fossil_use * 73.5) / 1e6
    line, = ax3.plot(net_co2.index, net_co2.values, linewidth=1.5,
                     color=continent_palette["Europe"], linestyle=ets_ls[region_id])
    legend_groups["Europe"].append((line, ets_labels[region_id]))
    ets_total = net_co2.copy() if ets_total is None else ets_total + net_co2

# Free allowances annotation
free_allowances = 20e6
ax3.axhline(y=free_allowances, color="black", alpha=0.5, linestyle="-", linewidth=1)
ax3.text(2019.5, free_allowances * 1.02, "Max ETS allowances for SAF purchase",
         color="black", alpha=0.7, fontsize=7, ha="left", va="bottom",
         bbox=dict(facecolor="white", edgecolor="none", alpha=0.6, pad=0.5))

# --- CORSIA by continent (INT regions only) ---
int_regions = [r for r in regions if "INT" in r]
corsia_by_continent = {}
corsia_total = None

for continent in continents_order:
    c_color = continent_palette[continent]
    c_int_regs = [r for r in int_regions if region_to_continent.get(r) == continent]
    if not c_int_regs:
        continue

    continent_offset = None
    for rid in c_int_regs:
        col_offset = f"{rid}:carbon_offset"
        if col_offset in vo.columns:
            s = vo[col_offset] / corsia_ratio * 1e6
            continent_offset = s.copy() if continent_offset is None else continent_offset + s

    if continent_offset is not None:
        line, = ax3.plot(continent_offset.index, continent_offset.values,
                         linewidth=1.5, color=c_color, linestyle=ls_corsia)
        legend_groups[continent].append((line, "CORSIA"))
        corsia_by_continent[continent] = continent_offset
        corsia_total = (continent_offset.copy() if corsia_total is None
                        else corsia_total + continent_offset)

# Europe CORSIA + ETS sum
if "Europe" in corsia_by_continent and ets_total is not None:
    eu_total = ets_total + corsia_by_continent["Europe"]
    line, = ax3.plot(eu_total.index, eu_total.values, linewidth=2,
                     color=continent_palette["Europe"], linestyle=ls_sum)
    legend_groups["Europe"].append((line, "CORSIA + ETS"))

# Total ETS
if ets_total is not None:
    line, = ax3.plot(ets_total.index, ets_total.values, linewidth=2,
                     color="black", linestyle=":")
    legend_totals.append((line, "Total ETS"))

# Total CORSIA
if corsia_total is not None:
    line, = ax3.plot(corsia_total.index, corsia_total.values, linewidth=2,
                     color="black", linestyle=ls_corsia)
    legend_totals.append((line, "Total CORSIA"))

# Global CORSIA + ETS
if corsia_total is not None and ets_total is not None:
    global_sum = corsia_total.add(ets_total, fill_value=0)
    line, = ax3.plot(global_sum.index, global_sum.values, linewidth=2,
                     color="black", linestyle=ls_sum)
    legend_totals.append((line, "Total CORSIA + ETS"))

# ── Build ordered legend: group by continent, then totals ──
handles, labels = [], []
for continent in continents_order:
    if not legend_groups[continent]:
        continue
    for h, l in legend_groups[continent]:
        handles.append(h)
        labels.append(f"{continent} — {l}")
for h, l in legend_totals:
    handles.append(h)
    labels.append(l)

ax3.legend(handles, labels, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax3.set_xlabel("Year")
ax3.set_xlim(2019, 2050)
ax3.set_ylabel("ETS Allowances / CORSIA Credits")
ax3.set_title("ETS Allowances & CORSIA Credits Required")
ax3.grid(True, alpha=0.3)
ax3.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
process.data["vector_outputs"]["overall:load_factor"]

In [ ]:
# LTAG refs 2 & 3 are now plotted directly in the main plot cell above.

In [ ]:
LTAG_series_3_yearly_scaled


### 4.1 Policy Contribution Treemap

Isolate the cumulative CO₂ reduction attributable to each policy lever, per region:
- **Efficiency** (fleet renewal + operations): `cumulative_co2_emissions_2019technology` − `cumulative_co2_emissions_including_load_factor`
- **Low-carbon energy** (SAF, hydrogen, electrification…): `cumulative_co2_emissions_including_load_factor` − `cumulative_co2_emissions`
- **CORSIA Offsets**: global `cumulative_carbon_offset`

Special groupings: USA_DOM + USA_INT → **SAF GC**, EU_DOM + EU_INT → **ReFuelEU**

In [ ]:
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np

pio.renderers.default = "vscode"

regions = process.list_regions()
vo = process.data["vector_outputs"]

# Region grouping / renaming rules
merge_groups = {
    "USA_DOM": "SAF GC",
    "USA_INT": "SAF GC",
    "EU_DOM": "ReFuelEU",
    "EU_INT": "ReFuelEU",
    "UK_DOM": "UK SAF",
    "BRAS_DOM": "BRA SAF",
    "SIN_INT": "SIN SAF",
}

rows = []
for region_id in regions:
    col_cumul_2019t = f"{region_id}:cumulative_co2_emissions_2019technology"
    col_cumul_lf = f"{region_id}:cumulative_co2_emissions_including_load_factor"
    col_cumul = f"{region_id}:cumulative_co2_emissions"
    col_offset = f"{region_id}:cumulative_carbon_offset"

    display_name = merge_groups.get(region_id, region_id)

    # Efficiency contribution (frozen 2019 tech → after operations/load factor)
    if col_cumul_2019t in vo.columns and col_cumul_lf in vo.columns:
        eff_reduction = (vo.loc[2050, col_cumul_2019t] - vo.loc[2050, col_cumul_lf]) / corsia_ratio
        if abs(eff_reduction) > 0.01:
            rows.append({
                "region": display_name,
                "policy": "Efficiency",
                "reduction_Gt": eff_reduction,
            })

    # Low-carbon energy contribution
    if col_cumul_lf in vo.columns and col_cumul in vo.columns:
        energy_reduction = (vo.loc[2050, col_cumul_lf] - vo.loc[2050, col_cumul]) / corsia_ratio
        if abs(energy_reduction) > 0.01:
            rows.append({
                "region": display_name,
                "policy": "Low-carbon energy",
                "reduction_Gt": energy_reduction,
            })

    # CORSIA offset contribution
    if col_offset in vo.columns:
        offset_val = vo.loc[2050, col_offset] / corsia_ratio
        if abs(offset_val) > 0.01:
            rows.append({
                "region": display_name,
                "policy": "CORSIA Offsets",
                "reduction_Gt": offset_val,
            })

df_policy = pd.DataFrame(rows)

# Aggregate merged groups (SAF GC, ReFuelEU, etc.)
df_policy = df_policy.groupby(["region", "policy"], as_index=False)["reduction_Gt"].sum()

# Keep only strictly positive values for treemap geometry
df_policy_plot = df_policy[df_policy["reduction_Gt"] > 0].copy()

# CORSIA shown as a single block (no region detail)
df_corsia = df_policy_plot[df_policy_plot["policy"] == "CORSIA Offsets"].copy()
df_corsia_total = pd.DataFrame([{
    "region": "All regions",
    "policy": "CORSIA Offsets",
    "reduction_Gt": df_corsia["reduction_Gt"].sum(),
}])

# Efficiency shown as a single block (no region detail)
df_eff = df_policy_plot[df_policy_plot["policy"] == "Efficiency"].copy()
df_eff_total = pd.DataFrame([{
    "region": "All regions",
    "policy": "Efficiency",
    "reduction_Gt": df_eff["reduction_Gt"].sum(),
}])

df_energy = df_policy_plot[df_policy_plot["policy"] == "Low-carbon energy"]
df_policy_plot = pd.concat([df_energy, df_eff_total, df_corsia_total], ignore_index=True)
df_policy_plot = df_policy_plot[df_policy_plot["reduction_Gt"] > 0]

fig = px.treemap(
    df_policy_plot,
    path=[px.Constant("All Reductions"), "policy", "region"],
    values="reduction_Gt",
    color="policy",
    color_discrete_map={
        "Efficiency": "#ff7f0e",
        "Low-carbon energy": "#2ca02c",
        "CORSIA Offsets": "#1f77b4",
    },
    title="Cumulative CO₂ reduction by policy and region (2020–2050)",
)

fig.update_traces(
    branchvalues="total",
    texttemplate="<b>%{label}</b><br>%{value:.1f} Gt",
    hovertemplate="<b>%{label}</b><br>Reduction: %{value:.1f} Gt<extra></extra>",
    tiling=dict(pad=4),
    marker=dict(line=dict(width=1.0, color="rgba(255,255,255,0.9)")),
)

fig.update_layout(
    margin=dict(t=70, l=10, r=10, b=10),
    height=550,
    uniformtext=dict(minsize=10, mode="show"),
)
fig.show()

# Summary table
print("\n📊 Policy contribution summary (cumulative Gt, 2020–2050):")
print("─" * 55)
for policy in ["Efficiency", "Low-carbon energy", "CORSIA Offsets"]:
    sub = df_policy[df_policy["policy"] == policy]
    if not sub.empty:
        total = sub["reduction_Gt"].sum()
        print(f"  {policy:22s}: {total:.1f} Gt")
print(f"  {'TOTAL':22s}: {df_policy['reduction_Gt'].sum():.1f} Gt")

df_policy_plot

## 5. Geographic Visualizations


### 5.1. Treemap: 2050 Emissions and Change vs 2019

Treemap encoding:
- **Area** = regional `co2_emissions_including_energy` in 2050
- **Color** = % change from 2019 to 2050
  - Green: decrease
  - Red: increase

In [ ]:
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np

# Stable renderer in VS Code notebooks
pio.renderers.default = "vscode"

# --- Mapping: AeroMAPS region → Continent ---
region_to_continent = {
    "AF_DOM": "Africa",
    "AF_INT": "Africa",
    "AS_DOM": "Asia",
    "OTHER_AS_INT": "Asia",
    "SIN_INT": "Asia",
    "EU_DOM": "Europe",
    "EU_INT": "Europe",
    "UK_DOM": "Europe",
    "UK_INT": "Europe",
    "OTHER_EUR_DOM": "Europe",
    "OTHER_EUR_INT": "Europe",
    "USA_DOM": "North America",
    "USA_INT": "North America",
    "OTHER_NAM_DOM": "North America",
    "OTHER_NAM_INT": "North America",
    "BRAS_DOM": "South America",
    "OTHER_SAM_DOM": "South America",
    "SAM_INT": "South America",
    "OC_DOM": "Oceania",
    "OC_INT": "Oceania",
}


def build_emissions_table(process_data, year):
    rows = []
    regions = process.list_regions()

    for region_id in regions:
        col = f"{region_id}:co2_emissions_including_energy"
        if col in process_data["vector_outputs"].columns and year in process_data["vector_outputs"].index:
            rows.append(
                {
                    "region": region_id,
                    "continent": region_to_continent.get(region_id, "Unknown"),
                    f"co2_{year}": float(process_data["vector_outputs"].loc[year, col]) / corsia_ratio,
                }
            )

    return pd.DataFrame(rows)


# Build and merge 2019/2050 data
df_2019 = build_emissions_table(process.data, 2019)
df_2050 = build_emissions_table(process.data, 2050)

df = df_2050.merge(df_2019[["region", "co2_2019"]], on="region", how="left")

if df.empty:
    print("No treemap data found. Check that the process was computed and years 2019/2050 exist.")
else:
    # Guard against zero/negative 2019 values for % evolution
    denom = df["co2_2019"].replace(0, np.nan)
    raw_delta_pct = ((df["co2_2050"] - df["co2_2019"]) / denom) * 100
    raw_delta_pct = raw_delta_pct.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Round evolution in the data itself
    df["delta_pct"] = np.rint(raw_delta_pct).astype(int)

    # IMPORTANT: use true plotting values for strict aggregation (no artificial +0.1)
    # Negative values are clipped to 0 because treemap areas must be non-negative
    df["co2_2050_plot"] = df["co2_2050"].clip(lower=0.0)

    # Keep only strictly positive regions for treemap geometry
    df_plot = df[df["co2_2050_plot"] > 0].copy()

    if df_plot.empty:
        print("No strictly positive 2050 emissions available for treemap areas.")
    else:
        max_abs_change = max(abs(df_plot["delta_pct"].min()), abs(df_plot["delta_pct"].max()))
        max_abs_change = max(float(max_abs_change), 1.0)

        fig = px.treemap(
            df_plot,
            path=[px.Constant("World"), "continent", "region"],
            values="co2_2050_plot",
            color="delta_pct",
            color_continuous_scale=[
                [0.0, "#1a9850"],
                [0.5, "#f7f7f7"],
                [1.0, "#d73027"],
            ],
            range_color=[-max_abs_change, max_abs_change],
            custom_data=["co2_2019", "co2_2050", "delta_pct"],
            title=(
                f"Treemap — Area: 2050 CO2 emissions | Color: change vs 2019 "
                f"(Total shown: {df_plot['co2_2050_plot'].sum():.1f} Mt)"
            ),
        )

        fig.update_traces(
            branchvalues="total",
            texttemplate="<b>%{label}</b><br>%{customdata[1]:.1f} Mt<br>%{customdata[2]:+d}%",
            textfont=dict(size=11),
            hovertemplate=(
                "<b>%{label}</b><br>"
                "2019: %{customdata[0]:.1f} Mt<br>"
                "2050: %{customdata[1]:.1f} Mt<br>"
                "Change: %{customdata[2]:+d}%<extra></extra>"
            ),
            tiling=dict(pad=4),
            marker=dict(line=dict(width=1.0, color="rgba(255,255,255,0.95)")),
        )

        try:
            fig.update_traces(marker=dict(cornerradius=6, line=dict(width=1.2, color="rgba(255,255,255,0.9)")))
        except Exception:
            pass

        fig.update_coloraxes(colorbar_title="Change vs 2019 (%)", colorbar_tickformat="+d")
        fig.update_layout(
            margin=dict(t=70, l=10, r=10, b=10),
            height=560,
            uniformtext=dict(minsize=11, mode="show"),
        )
        fig.show()

        # Quick consistency check: parent aggregation equals children sums
        check = df_plot.groupby("continent", as_index=False)["co2_2050_plot"].sum()
        print("\n✓ Aggregation check (sum of children by continent):")
        print(check)


# --- Summary by continent ---
print("\n📊 Emissions by Continent (2019 → 2050):")
print("─" * 62)
continent_summary = []
for continent in ["Asia", "North America", "Europe", "Africa", "South America", "Oceania"]:
    co2_2019 = df[df["continent"] == continent]["co2_2019"].sum()
    co2_2050 = df[df["continent"] == continent]["co2_2050"].sum()
    change_pct = ((co2_2050 - co2_2019) / co2_2019 * 100) if co2_2019 > 0 else 0.0
    change_pct = int(np.rint(change_pct))
    print(f"{continent:15s}: {co2_2019:7.1f} → {co2_2050:7.1f} Mt ({change_pct:+d}%)")
    continent_summary.append(
        {
            "continent": continent,
            "co2_2019": co2_2019,
            "co2_2050": co2_2050,
            "change_pct": change_pct,
        }
    )

# Save continent-level summary for optional external use
df_continent = pd.DataFrame(continent_summary)
df_continent.to_csv("data/continent_emissions.csv", index=False)
print("\n✓ Saved continent data to 'data/continent_emissions.csv'")

### 5.2 Geo Treemaps

In [ ]:
""" # 5.2: geo-inspired treemap + world map background
# Domain sizes are proportional to 2050 emissions across continents.

import plotly.graph_objects as go
import numpy as np
import json
import urllib.request
import math

if "df" not in globals() or df.empty:
    print("Please run section 5.1 first (it creates 'df').")
else:
    max_abs_change = max(abs(df["delta_pct"].min()), abs(df["delta_pct"].max()))
    max_abs_change = max(float(max_abs_change), 1.0)

    # Geographic centers for each continent (in [0,1]×[0,1] normalized sæpace)
    continent_centers = {
        "North America": (0.17, 0.78),
        "South America": (0.17, 0.34),
        "Europe":        (0.47, 0.78),
        "Africa":        (0.47, 0.46),
        "Asia":          (0.75, 0.74),
        "Oceania":       (0.84, 0.26),
    }

    # Compute 2050 emissions per continent (only positive regions)
    df_pos = df[df["co2_2050_plot"] > 0].copy()
    continent_emissions = df_pos.groupby("continent")["co2_2050_plot"].sum().to_dict()

    # Scale domains: area ∝ emissions, using sqrt for side length
    max_emission = max(continent_emissions.values()) if continent_emissions else 1.0
    # Max half-side for the largest continent (controls overall visual scale)
    MAX_HALF_SIDE = 0.28

    continent_domains = {}
    for continent, (cx, cy) in continent_centers.items():
        emi = continent_emissions.get(continent, 0.0)
        if emi <= 0:
            continue
        # half-side proportional to sqrt(emissions / max_emission)
        half = MAX_HALF_SIDE * math.sqrt(emi / max_emission)
        # Clamp to [0, 1] bounds
        x0 = max(0.0, cx - half)
        x1 = min(1.0, cx + half)
        y0 = max(0.0, cy - half)
        y1 = min(1.0, cy + half)
        continent_domains[continent] = dict(x=[x0, x1], y=[y0, y1])

    fig_geo_bg = go.Figure()

    fig_geo_bg.update_layout(
        xaxis=dict(range=[0, 1], visible=False),
        yaxis=dict(range=[0, 1], visible=False),
    )

    # --------------------------------------------------
    # World background via GeoJSON (stdlib only, no geopandas)
    # --------------------------------------------------
    GEOJSON_URL = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
    GEOJSON_URL_FALLBACK = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.geojson"

    def fetch_geojson(url):
        with urllib.request.urlopen(url, timeout=15) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        geojson = fetch_geojson(GEOJSON_URL)
        bg_source = "geo-countries (GitHub)"
    except Exception:
        try:
            geojson = fetch_geojson(GEOJSON_URL_FALLBACK)
            bg_source = "Natural Earth (naciscdn)"
        except Exception as e:
            geojson = None
            print(f"⚠ Could not load world GeoJSON: {e}")

    polygons_drawn = 0

    if geojson is not None:
        for feature in geojson.get("features", []):
            geom = feature.get("geometry")
            if geom is None:
                continue
            geom_type = geom.get("type", "")
            coords_list = geom.get("coordinates", [])

            if geom_type == "Polygon":
                polygons = [coords_list[0]]
            elif geom_type == "MultiPolygon":
                polygons = [poly[0] for poly in coords_list]
            else:
                continue

            for ring in polygons:
                try:
                    xs = [float((lon + 180.0) / 360.0) for lon, lat, *_ in ring]
                    ys = [float((lat + 90.0) / 180.0) for lon, lat, *_ in ring]
                    fig_geo_bg.add_trace(
                        go.Scatter(
                            x=xs,
                            y=ys,
                            mode="none",
                            fill="toself",
                            fillcolor="rgba(120,120,120,0.24)",
                            line=dict(width=0),
                            hoverinfo="skip",
                            showlegend=False,
                        )
                    )
                    polygons_drawn += 1
                except Exception:
                    continue

    print(f"✓ Background loaded from {bg_source} ({polygons_drawn} polygons)")

    # Add continent treemaps on top (sized proportionally to emissions)
    for idx, (continent, domain) in enumerate(continent_domains.items()):
        sub = df_pos[df_pos["continent"] == continent].copy()
        if sub.empty:
            continue

        child_values = sub["co2_2050_plot"].tolist()
        parent_value = sum(child_values)

        labels = [continent] + sub["region"].tolist()
        parents = [""] + [continent] * len(sub)
        values = [parent_value] + child_values

        cont_co2_2019 = sub["co2_2019"].sum()
        cont_co2_2050 = sub["co2_2050"].sum()
        cont_delta = int(np.rint((cont_co2_2050 - cont_co2_2019) / cont_co2_2019 * 100)) if cont_co2_2019 > 0 else 0
        texts = [f"{cont_co2_2050:.1f} Mt<br>{cont_delta:+d}%"] + [
            f"{v:.1f} Mt<br>{p:+d}%"
            for v, p in zip(sub["co2_2050"], sub["delta_pct"])
        ]
        colors = [cont_delta] + sub["delta_pct"].tolist()

        fig_geo_bg.add_trace(
            go.Treemap(
                labels=labels,
                parents=parents,
                values=values,
                branchvalues="total",
                domain=domain,
                marker=dict(
                    colors=colors,
                    colorscale=[
                        [0.0, "#1a9850"],
                        [0.5, "#f7f7f7"],
                        [1.0, "#d73027"],
                    ],
                    cmin=-max_abs_change,
                    cmax=max_abs_change,
                    line=dict(width=0.8, color="rgba(255,255,255,0.9)"),
                    colorbar=dict(title="Change vs 2019 (%)", tickformat="+d") if idx == 0 else None,
                    showscale=True if idx == 0 else False,
                ),
                customdata=[[cont_co2_2019, cont_co2_2050, cont_delta]] + list(
                    zip(sub["co2_2019"], sub["co2_2050"], sub["delta_pct"])
                ),
                text=texts,
                textinfo="label+text",
                hovertemplate=(
                    "<b>%{label}</b><br>"
                    "2019: %{customdata[0]:.1f} Mt<br>"
                    "2050: %{customdata[1]:.1f} Mt<br>"
                    "Change: %{customdata[2]:+d}%<extra></extra>"
                ),
                textfont=dict(size=10),
                tiling=dict(pad=3),
                opacity=0.82,
            )
        )

    fig_geo_bg.update_layout(
        title="Geo-inspired Treemap — Domain size ∝ 2050 CO₂ emissions",
        margin=dict(t=70, l=10, r=10, b=10),
        height=650,
        uniformtext=dict(minsize=9, mode="show"),
        paper_bgcolor="rgb(248,248,248)",
        plot_bgcolor="rgb(248,248,248)",
    )

    fig_geo_bg.show() """

## 6. Cumulative Emissions Analysis by Continent

Compare cumulative CO₂ emissions (2020–2050) per continent against the LTAG reference budget, using two downscaling methods:
- **Grandfathering**: budget proportional to 2019 emission share
- **Equitable (population)**: budget proportional to average population share (mean 2019/2050)

In [ ]:
import pandas as pd
import numpy as np

# --- 6.1 Cumulative net emissions by continent (2050 value) ---
regions = process.list_regions()

rows = []
for region_id in regions:
    col_cumul = f"{region_id}:cumulative_co2_emissions"
    col_no_energy = f"{region_id}:cumulative_co2_emissions_including_load_factor"
    col_offset = f"{region_id}:cumulative_carbon_offset"
    cumul_co2 = process.data["vector_outputs"].loc[2050, col_cumul] / corsia_ratio if col_cumul in process.data["vector_outputs"].columns else 0.0
    cumul_co2_no_energy = process.data["vector_outputs"].loc[2050, col_no_energy] / corsia_ratio if col_no_energy in process.data["vector_outputs"].columns else 0.0
    cumul_off = process.data["vector_outputs"].loc[2050, col_offset] / corsia_ratio if col_offset in process.data["vector_outputs"].columns else 0.0
    rows.append({
        "region": region_id,
        "continent": region_to_continent.get(region_id, "Unknown"),
        "cumul_co2": cumul_co2,
        "cumul_co2_no_energy": cumul_co2_no_energy,
        "cumul_offset": cumul_off,
        "cumul_net": cumul_co2 - cumul_off,
    })

df_cumul = pd.DataFrame(rows)
df_cumul_continent = df_cumul.groupby("continent")[["cumul_co2", "cumul_co2_no_energy", "cumul_offset", "cumul_net"]].sum()

print("Cumulative net CO₂ emissions by continent (Gt, 2020-2050):")
print("─" * 60)
for continent in ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]:
    r = df_cumul_continent.loc[continent]
    print(f"{continent:15s}: {r['cumul_co2']:8.1f} - {r['cumul_offset']:6.1f} offset = {r['cumul_net']:8.1f} Gt net")

global_cumul_net = df_cumul_continent["cumul_net"].sum()
print(f"\n{'GLOBAL':15s}: {global_cumul_net:.1f} Gt net cumulative")
df_cumul_continent

In [ ]:
# --- 6.2 LTAG cumulative budget vs scenario ---
# Cumulative LTAG emissions from 2020 to 2050 (sum of annual Mt values)
LTAG_cumul_budget = LTAG_series_3_yearly_scaled.loc[2020:2050].sum() / 1000.0  # Convert Mt to Gt


net_budget_1_5_50 = 500 # GT
carbon_removal_2100 = 527 # GT
aviation_grandfathering = 2.4 # %
gross_carbon_budget = (net_budget_1_5_50 + carbon_removal_2100) * (aviation_grandfathering / 100) # GT

# Global scenario cumulative (from overall aggregated outputs)
overall_cumul_co2 = process.data["vector_outputs"].loc[2050, "overall:cumulative_co2_emissions"] / corsia_ratio
overall_cumul_offset = process.data["vector_outputs"].loc[2050, "overall:cumulative_carbon_offset"] / corsia_ratio
scenario_cumul_net = overall_cumul_co2 - overall_cumul_offset

overshoot_pct = (scenario_cumul_net - LTAG_cumul_budget) / LTAG_cumul_budget * 100

print(f"LTAG cumulative budget (2020-2050):  {LTAG_cumul_budget:.1f} Gt")
print(f"PA aligned - aviation grandfathering budget:  {gross_carbon_budget:.1f} Gt")
print(f"Scenario cumulative net (2020-2050): {scenario_cumul_net:.1f} Gt")
print(f"Overshoot: {overshoot_pct:+.1f}%")

In [ ]:
# --- 6.3 Downscaling: Grandfathering vs Equitable (population) ---

# a) Grandfathering: share of 2019 emissions
emissions_2019_by_continent = {}
for region_id in regions:
    col = f"{region_id}:co2_emissions_including_energy"
    if col in process.data["vector_outputs"].columns:
        continent = region_to_continent.get(region_id, "Unknown")
        emissions_2019_by_continent[continent] = emissions_2019_by_continent.get(continent, 0.0) + \
            process.data["vector_outputs"].loc[2019, col]

total_emissions_2019 = sum(emissions_2019_by_continent.values())
grandfathering_shares = {c: v / total_emissions_2019 for c, v in emissions_2019_by_continent.items()}

# b) Equitable: share of average population (mean 2019/2050)
population_data = {
    "Africa":        {"pop_2019": 1_348_005_692, "pop_2050": 2_466_647_773},
    "Asia":          {"pop_2019": 4_651_098_724, "pop_2050": 5_278_869_940},
    "Europe":        {"pop_2019":   750_809_114, "pop_2050":   704_554_531},
    "North America": {"pop_2019":   594_346_844, "pop_2050":   687_962_540},
    "Oceania":       {"pop_2019":    43_504_534, "pop_2050":    57_688_455},
    "South America": {"pop_2019":   423_548_291, "pop_2050":   468_674_525},
}
pop_avg = {c: (d["pop_2019"] + d["pop_2050"]) / 2 for c, d in population_data.items()}
total_pop_avg = sum(pop_avg.values())
equitable_shares = {c: v / total_pop_avg for c, v in pop_avg.items()}

# Build comparison table
comparison_rows = []
continents_order = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]

for continent in continents_order:
    actual = df_cumul_continent.loc[continent, "cumul_net"]
    
    budget_gf = LTAG_cumul_budget * grandfathering_shares[continent]
    overshoot_gf = (actual - budget_gf) / budget_gf * 100
    
    budget_eq = LTAG_cumul_budget * equitable_shares[continent]
    overshoot_eq = (actual - budget_eq) / budget_eq * 100
    
    comparison_rows.append({
        "Continent": continent,
        "CO2 cumul. net (Gt)": round(actual, 1),
        "GF share (%)": round(grandfathering_shares[continent] * 100, 1),
        "GF budget (Gt)": round(budget_gf, 1),
        "GF overshoot (%)": round(overshoot_gf, 1),
        "Pop share (%)": round(equitable_shares[continent] * 100, 1),
        "Pop budget (Gt)": round(budget_eq, 1),
        "Pop overshoot (%)": round(overshoot_eq, 1),
    })

df_comparison = pd.DataFrame(comparison_rows)
print("Downscaling comparison — Grandfathering vs Equitable (population)")
print("=" * 90)
df_comparison

In [ ]:
# --- 6.4 Visualization: Circular bar plot (Planetary Boundaries style) ---
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

actual_vals = np.array([df_cumul_continent.loc[c, "cumul_net"] for c in continents_order])
actual_vals_no_offsets = np.array([df_cumul_continent.loc[c, "cumul_co2"] for c in continents_order])
gf_vals = np.array([LTAG_cumul_budget * grandfathering_shares[c] for c in continents_order])
eq_vals = np.array([LTAG_cumul_budget * equitable_shares[c] for c in continents_order])

n = len(continents_order)
wedge_angle = 2 * np.pi / n
separator_half_width = 0.005

r_budget = 1.0

r_abs_budget_1_5 = gross_carbon_budget / LTAG_cumul_budget

r_cap_eq = 4

fig, (ax_gf, ax_eq) = plt.subplots(1, 2, figsize=(14, 7),
                                     subplot_kw=dict(projection="polar"))

for ax, budget_vals, title, use_cap in [
    (ax_gf, gf_vals, "2019 emissions grandfathering \n (ATAG S1 deviation)", False),
    (ax_eq, eq_vals, "2019-2050 population-based  \n (ATAG S1 deviation)", True),
]:
    ratios = actual_vals / budget_vals
    ratios_no_offsets = actual_vals_no_offsets / budget_vals
    r_max_actual = ratios_no_offsets.max()

    if use_cap:
        r_outer = r_cap_eq + 0.55
    else:
        r_outer = r_max_actual + 0.55

    for i in range(n):
        theta_start = i * wedge_angle + separator_half_width
        theta_end = (i + 1) * wedge_angle - separator_half_width
        thetas = np.linspace(theta_start, theta_end, 80)

        r_actual = ratios[i]
        r_actual_no_offsets = ratios_no_offsets[i]

        if r_actual <= r_budget:
            ax.fill_between(thetas, 0, r_actual, color="#1a9850", alpha=0.7)
            ax.fill_between(thetas, r_actual, r_budget, color="#1a9850", alpha=0.15)
            ax.plot(
                    thetas,
                    np.full_like(thetas, r_actual_no_offsets, dtype=float),
                    color="#1a9850",
                    linewidth=1,
                    linestyle="--",
                    alpha=0.7,
                )
        else:
            ax.fill_between(thetas, 0, r_budget, color="#1a9850", alpha=0.7)
            if not use_cap or r_actual <= r_cap_eq:
                ax.fill_between(thetas, r_budget, r_actual, color="#d73027", alpha=0.65)
                ax.plot(
                    thetas,
                    np.full_like(thetas, r_actual_no_offsets, dtype=float),
                    color="#d73027",
                    linewidth=1,
                    linestyle="--",
                    alpha=0.7,
                )
            else:
                ax.fill_between(thetas, r_budget, r_cap_eq, color="#d73027", alpha=0.65, linewidth=0)
                n_fade = 15
                fade_r = np.linspace(r_cap_eq, r_cap_eq + 0.45, n_fade + 1)
                for j in range(n_fade):
                    alpha_fade = 0.45 * (1 - j / n_fade)
                    ax.fill_between(thetas, fade_r[j], fade_r[j + 1],
                                    color="#d73027", alpha=alpha_fade)

        # Separator lines
        for theta_boundary in [i * wedge_angle, (i + 1) * wedge_angle]:
            ax.plot([theta_boundary, theta_boundary], [0, r_outer - 0.1],
                    color="black", linewidth=1.2, alpha=0.75)

        # Horizontal text label at the midpoint of the wedge
        overshoot_pct = (r_actual - 1) * 100
        mid_theta = (theta_start + theta_end) / 2
        label_text = f"{continents_order[i]}\n({overshoot_pct:+.0f}%)"

        ax.text(mid_theta, r_outer - 0.08, label_text,
                ha="center", va="center", fontsize=7.5, fontweight="bold",
                rotation=0)

    # Budget reference ring
    ring_thetas = np.linspace(0, 2 * np.pi, 300)
    ax.plot(ring_thetas, np.full_like(ring_thetas, r_budget), color="black",
            linewidth=1.8, linestyle="--", alpha=0.7)
    ax.plot(ring_thetas, np.full_like(ring_thetas, r_abs_budget_1_5), color="#2f5cc6",
            linewidth=1.8, linestyle="--", alpha=0.7)

    ax.set_ylim(0, r_outer + 0.7)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.grid(False)
    ax.spines['polar'].set_visible(False)
    ax.set_title(title, fontsize=12)

# Legend
safe_patch = mpatches.Patch(color="#1a9850", alpha=0.7, label="Within budget")
over_patch = mpatches.Patch(color="#d73027", alpha=0.65, label="Beyond budget")
no_ofst_line = plt.Line2D([0], [0], color="#d73027", linewidth=1, linestyle="--", label="Without offsets")
budget_line = plt.Line2D([0], [0], color="black", linewidth=1.5, linestyle="--", label="ATAG S1 budget")
abs_budget_line = plt.Line2D([0], [0], color="#2f5cc6", linewidth=1.5, linestyle="--", label="2100 1.5°C budget (aviation grandfathering)")
fig.legend(handles=[safe_patch, over_patch, no_ofst_line, budget_line, abs_budget_line],
           loc="lower center", ncol=5, fontsize=10, frameon=False)

# fig.suptitle("Cumulative CO₂ emissions vs ATAG S1 budget by continent (2020–2050)",
#             fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

## 7. Sensitivity Analysis

Two alternative scenarios to test the robustness of the baseline multi-regional results:

1. **7.1 — USA without SAF**: Re-run the multi-regional model with the SAF mandate zeroed out for USA_DOM and USA_INT
2. **7.2 — Global ReFuelEU**: Run a standard (non-regional) AeroMAPS process where the ReFuelEU-inspired SAF mandate applies worldwide

### 7.1 USA without SAF mandate

Neutralise the SAF Grand Challenge mandate for USA_DOM and USA_INT by running a mini multi-regional process (2 regions only) with zeroed-out SAF quantities, then compare with the reference results.

In [ ]:
import time
from aeromaps import create_multi_regional_process

# Run a mini multi-regional process with only USA_DOM and USA_INT (zeroed SAF)
process_usa_no_saf = create_multi_regional_process(
    configuration_file="data/sensitivity_usa_no_saf/regionalisation_usa_no_saf.yaml",
    disable_execution_statistics=True,
)
start_time = time.time()
process_usa_no_saf.compute(parallel=False)
elapsed = time.time() - start_time
print(f"✓ Scenario 'USA no SAF' completed in {elapsed:.2f}s (2 regions only)")

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

vo_ref = process.data["vector_outputs"]
vo_nosaf = process_usa_no_saf.data["vector_outputs"]

# Reconstruct "global no-SAF" by replacing USA contributions in the reference
# Global_no_saf = Global_ref - USA_ref + USA_no_saf

usa_ref = (
    vo_ref["USA_DOM:co2_emissions_including_energy"]
    + vo_ref["USA_INT:co2_emissions_including_energy"]
) / corsia_ratio 

usa_ref_w_off = (
    vo_ref["USA_DOM:co2_emissions_including_energy"]
    + vo_ref["USA_INT:co2_emissions_including_energy"] - vo_ref["USA_INT:carbon_offset"] - vo_ref["USA_DOM:carbon_offset"]
) / corsia_ratio

usa_nosaf = (vo_nosaf["overall:co2_emissions_including_energy"]) / corsia_ratio
usa_nosaf_w_off = (vo_nosaf["overall:co2_emissions_including_energy"] - vo_nosaf["USA_INT:carbon_offset"]) / corsia_ratio

ref_global = (vo_ref["overall:co2_emissions_including_energy"]) / corsia_ratio
ref_global_w_off = (vo_ref["overall:co2_emissions_including_energy"] - vo_ref["overall:carbon_offset"]) / corsia_ratio

nosaf_global = ref_global - usa_ref + usa_nosaf
nosaf_global_w_off = ref_global_w_off - usa_ref_w_off + usa_nosaf_w_off

# --- Left: Global emissions comparison ---
ref_global.plot(ax=ax1, label="Reference", linewidth=2.5, color="#2ca02c")
ref_global_w_off.plot(ax=ax1, label="Reference (with offsets)", linewidth=2.5, color="#2ca02c", linestyle="--")
nosaf_global.plot(ax=ax1, label="USA no SAF", linewidth=2.5, color="#d62728", linestyle="-")
nosaf_global_w_off.plot(ax=ax1, label="USA no SAF (with offsets)", linewidth=2.5, color="#d62728", linestyle="--")
if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(ax=ax1, label="ATAG S1", linewidth=2, color="black", linestyle=":")

ax1.set_xlabel("Year")
ax1.set_ylabel("CO₂ Emissions (Mt)")
ax1.set_title("Global CO₂ emissions")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Right: USA only (DOM + INT) ---
usa_ref.plot(ax=ax2, label="Reference", linewidth=2.5, color="#2ca02c")
usa_ref_w_off.plot(ax=ax2, label="Reference (with offsets)", linewidth=2.5, color="#596559", linestyle="--")
usa_nosaf.plot(ax=ax2, label="USA no SAF", linewidth=2.5, color="#d62728", linestyle="--")
usa_nosaf_w_off.plot(ax=ax2, label="USA no SAF (with offsets)", linewidth=2.5, color="#ff7f0e", linestyle="--")

ax2.set_xlabel("Year")
ax2.set_ylabel("CO₂ Emissions (Mt)")
ax2.set_title("USA (DOM + INT) CO₂ emissions")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Cumulative difference (reconstructed global)
ref_cumul_global = vo_ref.loc[2050, "overall:cumulative_co2_emissions"] / corsia_ratio
usa_ref_cumul = (
    vo_ref.loc[2050, "USA_DOM:cumulative_co2_emissions"]
    + vo_ref.loc[2050, "USA_INT:cumulative_co2_emissions"]
) / corsia_ratio
usa_nosaf_cumul = vo_nosaf.loc[2050, "overall:cumulative_co2_emissions"] / corsia_ratio
delta = usa_nosaf_cumul - usa_ref_cumul

fig.suptitle(
    f"Sensitivity: USA without SAF mandate — Δ cumulative = +{delta:.1f} Gt",
    fontsize=12, fontweight="bold",
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

### 7.2 Global ReFuelEU mandate

Run the full multi-regional AeroMAPS process (20 regions), but replace every region's energy carriers with a single ReFuelEU-inspired SAF mandate applied uniformly.

Step mandate (valid until preceding year of next step):
- 2% from 2025, 6% from 2030, 20% from 2035, 34% from 2040, 42% from 2045, 70% in 2050

In [ ]:
import time
from aeromaps import create_multi_regional_process

# Run the full multi-regional process with ReFuelEU energy carriers for every region
process_refueleu = create_multi_regional_process(
    configuration_file="data/sensitivity_global_refueleu/regionalisation_refueleu.yaml",
    disable_execution_statistics=True,
)

process_refueleu_enhanced = create_multi_regional_process(
    configuration_file="data/sensitivity_global_enhanced_refueleu/regionalisation_refueleu.yaml",
    disable_execution_statistics=True,
)

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

fig, ax1 = plt.subplots(figsize=(10, 5))

vo_ref = process.data["vector_outputs"]
vo_rfeu = process_refueleu.data["vector_outputs"]
vo_rfeu_enhanced = process_refueleu_enhanced.data["vector_outputs"]

# --- Left: Annual CO2 emissions ---
ref_global = vo_ref["overall:co2_emissions_including_energy"] / corsia_ratio
ref_global.plot(ax=ax1, label="Multi-regional ref.", linewidth=2, color="#2ca02c")

ref_global_w_off = (vo_ref["overall:co2_emissions_including_energy"] - vo_ref["overall:carbon_offset"]) / corsia_ratio
ref_global_w_off.plot(ax=ax1, label="Multi-regional ref. (with offset)", linewidth=2, linestyle="--", color="#2ca02c")

rfeu_global = vo_rfeu["overall:co2_emissions_including_energy"] / corsia_ratio
rfeu_global.plot(ax=ax1, label="Global ReFuelEU", linewidth=2, color="#9467bd")

rfeu_global_w_off = (vo_rfeu["overall:co2_emissions_including_energy"] - vo_rfeu["overall:carbon_offset"]) / corsia_ratio
rfeu_global_w_off.plot(ax=ax1, label="Global ReFuelEU (with offset)", linewidth=2, linestyle="--", color="#9467bd")

rfeu_global_enhanced = vo_rfeu_enhanced["overall:co2_emissions_including_energy"] / corsia_ratio
rfeu_global_enhanced.plot(ax=ax1, label="Global ReFuelEU (enhanced)", linewidth=2, color="#8c564b")

rfeu_global_enhanced_w_off = (vo_rfeu_enhanced["overall:co2_emissions_including_energy"] - vo_rfeu_enhanced["overall:carbon_offset"]) / corsia_ratio
rfeu_global_enhanced_w_off.plot(ax=ax1, label="Global ReFuelEU (enhanced, with offset)", linewidth=2, linestyle="--", color="#8c564b")

if "LTAG_series_3_yearly_scaled" in globals():
    LTAG_series_3_yearly_scaled.plot(ax=ax1, label="ATAG S1", linewidth=2, color="black", linestyle=":")

if "LTAG_series_6_yearly_scaled" in globals():
    LTAG_series_6_yearly_scaled.plot(ax=ax1, label="ATAG S1 (no offset)", linewidth=2, color="black", linestyle="--")

ax1.set_xlabel("Year")
ax1.set_ylabel("CO₂ Emissions (Mt)")
ax1.set_title("Annual CO₂ emissions")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
# --- Replace right subplot with cumulative values as text on left plot ---
ref_cumul = (vo_ref["overall:cumulative_co2_emissions"] - vo_ref["overall:cumulative_carbon_offset"]) / corsia_ratio
rfeu_cumul = (vo_rfeu["overall:cumulative_co2_emissions"] - vo_rfeu["overall:cumulative_carbon_offset"]) / corsia_ratio
rfeu_enhanced_cumul = (vo_rfeu_enhanced["overall:cumulative_co2_emissions"] - vo_rfeu_enhanced["overall:cumulative_carbon_offset"]) / corsia_ratio

total_offset_ref = vo_ref["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio
total_offset_rfeu = vo_rfeu["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio
total_offset_rfeu_enhanced = vo_rfeu_enhanced["overall:cumulative_carbon_offset"].iloc[-1] / corsia_ratio


# Hide unused right axis
ax2.axis("off")

# Add cumulative summary directly on the left chart
summary_txt = (
    f"Cumulative CO₂ in 2050\n"
    f"Ref: {ref_cumul.iloc[-1]:.1f} Gt, incl. {total_offset_ref:.1f} Gt offset\n"
    f"Global ReFuelEU: {rfeu_cumul.iloc[-1]:.1f} Gt, incl. {total_offset_rfeu:.1f} Gt offset\n"
    f"Global ReFuelEU (enhanced): {rfeu_enhanced_cumul.iloc[-1]:.1f} Gt, incl. {total_offset_rfeu_enhanced:.1f} Gt offset\n"
)
ax1.text(
    0.02, 0.98, summary_txt,
    transform=ax1.transAxes,
    va="top", ha="left",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85, edgecolor="0.8")
)

fig.suptitle(
    f"Sensitivity: Global ReFuelEU mandate",
    fontsize=12, fontweight="bold",
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

print(f"\nCumulative CO₂ at 2050:")
print(f"  Multi-regional ref.: {ref_cumul.iloc[-1]:.1f} Gt")
print(f"  Global ReFuelEU:     {rfeu_cumul.iloc[-1]:.1f} Gt")